# Astro 1221: Project 3 - Measuring the Expanding Universe
This notebook walks through the process of loading, cleaning, and analyzing Type Ia Supernova data to measure the expansion of the universe. We will use the classes defined in the `src` directory to perform the following steps:

1.  **Data Loading & Processing**: Fetch raw supernova data, parse it, and clean it into a usable format.
2.  **Standardization**: Apply the Tripp Relation to standardize the supernova apparent magnitudes, turning them into reliable "standard candles."
3.  **Cosmological Modeling**: Calculate theoretical distances using different models of the universe (a simple linear Hubble model vs. a more complex Lambda-CDM model).
4.  **Analysis & Visualization**: Compare the observed data against the theoretical models to find the best-fit cosmological parameters.

This notebook aims to replicate the core logic of the `Home.py` Streamlit application, providing a step-by-step scientific narrative.

## Step 1: Load and Process the Data

First, we'll import the necessary libraries and classes. We'll use `SupernovaDataLoader` to fetch the data and `SupernovaDataProcessor` to clean it up. We will also import `plotly.express` for plotting.

In [1]:
import pandas as pd
import plotly.express as px
from src.data_loader import SupernovaDataLoader
from src.data_processor import SupernovaDataProcessor
from src.optimizer import SupernovaOptimizer
# Execute the data loading and processing pipeline
try:
    loader = SupernovaDataLoader()
    processor = SupernovaDataProcessor()

    loader.fetch_and_save_json()
    raw_records = loader.parse_supernova_objects()
    supernova_df = processor.process_raw_records(raw_records)

    z = supernova_df['redshift']
    mu = supernova_df['mu']
    
    optimizer = SupernovaOptimizer()
    simple_fit = optimizer.fit_simple_hubble_model(z,mu)


    print(f"Successfully loaded and processed {len(supernova_df)} supernova records.")
    supernova_df.head()

except Exception as e:
    print(f"An error occurred: {e}")

Successfully loaded and processed 732 supernova records.


## Step 2: Standardize the Supernovae
Now that we have clean data, we need to standardize the supernovae. Type Ia supernovae are not all intrinsically the same brightness. Their light curves (how their brightness changes over time) can be "stretched" and their color can vary. The **Tripp Relation** is an empirical formula that corrects for these variations, allowing us to calculate a standardized distance modulus, `μ`.

The formula, as implemented in `SupernovaCosmologyModels.calculate_distance_modulus()`, is:

μ = m + (α * s) - (β * c) - M

Where:
- `m` is the observed apparent magnitude.
- `s` is the stretch factor of the light curve.
- `c` is the color value.
- `α` and `β` are empirically derived constants. For this project, we use `α = 0.14` and `β = 3.1`.
- `M` is the absolute magnitude of a Type Ia supernova, which is approximately `-19.3`.

Let's use our `SupernovaCosmologyModels` class to apply this correction.

In [2]:
from src.models import SupernovaCosmologyModels
from src.constants import SUPERNOVA_KEYS

# Initialize the cosmology models
cosmology = SupernovaCosmologyModels()

# Calculate the distance modulus for each supernova
supernova_df['distance_modulus'] = cosmology.calculate_distance_modulus(
    observedMagnitude=supernova_df[SUPERNOVA_KEYS['MAGNITUDE']],
    stretchFactor=supernova_df[SUPERNOVA_KEYS['STRETCH']],
    colorValue=supernova_df[SUPERNOVA_KEYS['COLOR']]
)

print("DataFrame with calculated distance modulus:")
supernova_df.head()

DataFrame with calculated distance modulus:


,supernova name,redshift,magnitude,stretch,color,mu,distance_modulus
0,03D1au,0.503084,23.001698,1.273191,-0.012353,42.301698,42.378239
1,03D1aw,0.580724,23.573937,0.974346,-0.025076,42.873937,42.948081
2,03D1ax,0.494795,22.960139,-0.728837,-0.099683,42.260139,42.327119
3,03D1bp,0.345928,22.398137,-1.155110,-0.040581,41.698137,41.522223
4,03D1co,0.677662,24.078115,0.618820,-0.039380,43.378115,43.446828


## Step 3: Cosmological Modeling - The Hubble Law

The Hubble Law is the simplest model of the universe's expansion. It states that the recessional velocity of a galaxy is directly proportional to its distance from us. For low redshifts (z < 0.1), this can be expressed with a linear relationship. Our `SupernovaCosmologyModels` class implements this with the `calculate_simple_hubble_model` function.

Let's break down how it works. The function performs two key steps:

**1. Calculate Luminosity Distance (`d_L`)**

First, it calculates the luminosity distance using the formula:

$d_L = (c * z) / H_0$

Where:
- `c` is the speed of light (`SPEED_OF_LIGHT_KM_S`).
- `z` is the redshift.
- `H_0` is the Hubble Constant.

**2. Convert Distance to Distance Modulus (`μ`)**

Next, it converts this physical distance in megaparsecs into the logarithmic distance modulus scale using the private helper function `_convert_distance_to_modulus`. The formula for this is:

$μ = 5 * log_{10}(d_L) + 25$

This formula is a standard in astronomy for relating the logarithmic magnitude system to physical distances.

Let's see a quick example with a single data point before plotting everything.

In [3]:
# Let's demonstrate with a single redshift value, z = 0.1
z_example = 0.1
H0 = 70.0

# 1. Calculate Luminosity Distance
from src.constants import SPEED_OF_LIGHT_KM_S
luminosity_distance_example = (SPEED_OF_LIGHT_KM_S * z_example) / H0
print(f"For z = {z_example}, the luminosity distance (d_L) is: {luminosity_distance_example:.2f} Mpc")

# 2. Convert to Distance Modulus
from src.constants import DISTANCE_MODULUS_LOG_MULTIPLIER, DISTANCE_MODULUS_OFFSET
mu_example_manual = DISTANCE_MODULUS_LOG_MULTIPLIER * np.log10(luminosity_distance_example) + DISTANCE_MODULUS_OFFSET
print(f"Manually calculated distance modulus (μ) is: {mu_example_manual:.2f}")

# Now, let's verify with our function from the class
mu_example_function = cosmology.calculate_empty_universe_model(z_example, H0)
print(f"Distance modulus (μ) from calculate_empty_universe_model() is: {mu_example_function:.2f}")

For z = 0.1, the luminosity distance (d_L) is: 428.27 Mpc


NameError: name 'np' is not defined

In [ ]:
import numpy as np

# Define a Hubble Constant from optimizer module
H0 = 73.35

# Calculate the theoretical model
z_range = np.linspace(supernova_df[SUPERNOVA_KEYS['REDSHIFT']].min(), supernova_df[SUPERNOVA_KEYS['REDSHIFT']].max(), 100)
simple_model_mu = cosmology.calculate_empty_universe_model(z_range, H0)

# Create the plot
fig = px.scatter(supernova_df, x=SUPERNOVA_KEYS['REDSHIFT'], y='distance_modulus', 
                 title='Supernova Distance Modulus vs. Redshift (Hubble Law)',
                 labels={SUPERNOVA_KEYS['REDSHIFT']: 'Redshift (z)', 'distance_modulus': 'Distance Modulus (μ)'})

# Add the theoretical line
fig.add_scatter(x=z_range, y=simple_model_mu, mode='lines', name=f'Simple Hubble Model (H₀={H0})')
fig.show()

## Step 4: Advanced Modeling - The Accelerating Universe (ΛCDM)

As you can see from the plot above, the simple Hubble Law fits the data well at low redshifts, but at higher redshifts (further distances), the observed supernovae are dimmer (have a larger distance modulus) than the model predicts. This was the Nobel Prize-winning discovery that indicates the universe's expansion is accelerating.

To model this, we need a more sophisticated model that includes terms for matter density (`Ω_m`) and dark energy (`Ω_Λ`). This is known as the Lambda-CDM (ΛCDM) model. The luminosity distance in this model is calculated by integrating over the expansion history of the universe, as described in Hogg (1999).

The core of this calculation is the expansion factor, `E(z)`:

E(z) = sqrt( Ω_m * (1 + z)^3 + Ω_Λ )

The `SupernovaCosmologyModels.calculate_advanced_cosmological_model` function implements this complex calculation. Let's now plot this model and compare it to our data and the simple Hubble model. We will use the standard "concordance" values of `Ω_m = 0.3` and `Ω_Λ = 0.7`.

In [4]:
# Define cosmological parameters for the ΛCDM model
H0 = 73.35
Omega_M = 0.3
Omega_L = 0.7

# Calculate the advanced model
advanced_model_mu = cosmology.calculate_advanced_cosmological_model(
    redshiftValues=z_range,
    hubbleConstant=H0,
    matterDensity=Omega_M,
    darkEnergyDensity=Omega_L
)

# Create the plot with all data
fig = px.scatter(supernova_df, x=SUPERNOVA_KEYS['REDSHIFT'], y='distance_modulus',
                 title='Supernova Data vs. Cosmological Models',
                 labels={SUPERNOVA_KEYS['REDSHIFT']: 'Redshift (z)', 'distance_modulus': 'Distance Modulus (μ)'})

# Add the simple model
fig.add_scatter(x=z_range, y=simple_model_mu, mode='lines', name=f'Simple Hubble Model (H₀={H0})')

# Add the advanced model
fig.add_scatter(x=z_range, y=advanced_model_mu, mode='lines', name=f'ΛCDM Model (Ωm={Omega_M}, ΩΛ={Omega_L})')

fig.show()

NameError: name 'z_range' is not defined

## Conclusion

The final plot clearly shows that the ΛCDM model, which incorporates dark energy, is a much better fit for the observed supernova data, especially at higher redshifts. The data points consistently fall closer to the ΛCDM line than the simple Hubble Law line. This discrepancy is the primary evidence for the accelerating expansion of the universe.

This notebook has demonstrated the end-to-end process of using Type Ia supernovae as standard candles to probe the nature of our universe, from raw data processing to fitting advanced cosmological models. The results align with the findings of modern cosmology.

**EVERY SECTION AFTER THIS IS LE3FTOVER FROM SOME OLD NOTES BUT ALREADY HAVE THE NOTATION FROM THE FORMULAS SO USE THOSE!!!*

## 3. Define the Astrophysical Model

We are fitting a Hubble diagram to the supernova data. First, we must standardize the apparent magnitude of the supernovae to calculate the **distance modulus**. This is done using the Tripp relation:

$$ \mu = m + (\alpha \cdot s) - (\beta \cdot c) - M $$

where:
- $\mu$ is the distance modulus
- $m$ is the observed apparent magnitude
- $s$ is the stretch factor of the supernova light curve
- $c$ is the color value
- $\alpha$ and $\beta$ are empirical constants
- $M$ is the absolute magnitude of a standard Type Ia supernova

The relationship between a supernova's distance modulus and its redshift is described by the following equation:

$$ \mu = 5 \log_{10}(d_L) + 25 $$

where $d_L$ is the luminosity distance in Megaparsecs (Mpc). For a simple, linearly expanding universe (at low redshift), the luminosity distance is related to redshift $z$ by:

$$ d_L(z) = \frac{c}{H_0} z $$

where $c$ is the speed of light and $H_0$ is the Hubble Constant. Our `src/models.py` file contains the `SupernovaCosmologyModels` class which implements these calculations.

## 4. Define the Fitting Methodology

To find the best-fit value for the Hubble Constant $H_0$, we will use a least-squares fitting approach. We want to minimize the chi-squared ($\chi^2$) statistic, which is a measure of the goodness of fit between our model and the data. The $\chi^2$ is defined as:

$$ \chi^2 = \sum_{i=1}^{N} \left( \frac{y_i - f(x_i; \vec{\theta})}{\sigma_i} \right)^2 $$

In our case, $y_i$ is the observed distance modulus, $f(x_i; \vec{\theta})$ is our `hubble_diagram_model` with parameters $\vec{\theta} = (H_0)$, $x_i$ is the redshift, and $\sigma_i$ is the uncertainty on the distance modulus. We will use `scipy.optimize.curve_fit` to perform this minimization.


I DON'T REMEMBER HOW ACCURATRE THIS IS IN OUR IMPLEMENTATION YET

In [5]:
results = optimizer.fit_advanced_cosmological_model(z, mu)

print("--- ADVANCED COSMOLOGICAL MODEL RESULTS ---")
print(f"Hubble Constant (H0):     {results['hubbleConstant']:.4f} ± {results['uncertainties']['H0_error']:.4f} km/s/Mpc")
print(f"Matter Density (0m):      {results['matterDensity']:.4f} ± {results['uncertainties']['0m_error']:.4f}")
print(f"Dark Energy Density (0l): {results['darkEnergyDensity']:.4f} ± {results['uncertainties']['0l_error']:.4f}")

print("\n--- COVARIANCE MATRIX ---")
print(results['covariance_matrix'])

# Quick Physics Check: Total Density
total_density = results['matterDensity'] + results['darkEnergyDensity']
print(f"\nTotal Density (0m + 0l):  {total_density:.4f}")

--- ADVANCED COSMOLOGICAL MODEL RESULTS ---
Hubble Constant (H0):     73.3500 ± 0.0000 km/s/Mpc
Matter Density (0m):      0.1601 ± 0.0190
Dark Energy Density (0l): 0.7000 ± 0.0302

--- COVARIANCE MATRIX ---
[[ 0.00035952 -0.0005429 ]
 [-0.0005429   0.00091363]]

Total Density (0m + 0l):  0.8601


In [7]:
results = optimizer.calculate_goodness_of_fit(mu, luminosityDistancesMpc, numberOfParameters)

# Access your results
print(f"Chi-Squared: {results['chisquaredvalue']}")
print(f"Reduced Chi-Squared: {results['reducedchisq']}")

NameError: name 'luminosityDistancesMpc' is not defined